# Tournament Simulation
Pada tahap ini dilakukan simulasi turnamen World Cup 2026 menggunakan model machine learning yang telah dilatih. Simulasi dilakukan mulai dari fase grup hingga final untuk mendapatkan estimasi peluang juara setiap negara.

## Import Library

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
from itertools import combinations
from collections import defaultdict

import matplotlib.pyplot as plt

In [2]:
## Load Model dan Dataset
model_path = Path("../model/worldcup_match_model.pkl")
data_path = Path("../data/processed/match_processed.csv")

model = joblib.load(model_path)

df = pd.read_csv(data_path)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

print("Model berhasil dimuat dari:", model_path)
print("Dataset berhasil dimuat dari:", data_path)

df.head()

Model berhasil dimuat dari: ..\model\worldcup_match_model.pkl
Dataset berhasil dimuat dari: ..\data\processed\match_processed.csv


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,result
0,2000-01-04,Egypt,Togo,2.0,1.0,Friendly,Aswan,Egypt,False,2000,home_win
1,2000-01-07,Tunisia,Togo,7.0,0.0,Friendly,Tunis,Tunisia,False,2000,home_win
2,2000-01-08,Trinidad and Tobago,Canada,0.0,0.0,Friendly,Port of Spain,Trinidad and Tobago,False,2000,draw
3,2000-01-09,Burkina Faso,Gabon,1.0,1.0,Friendly,Ouagadougou,Burkina Faso,False,2000,draw
4,2000-01-09,Guatemala,Armenia,1.0,1.0,Friendly,Los Angeles,United States,True,2000,draw


## Membuat Data Riwayat Performa Tim

In [3]:
home_data = df[[
    "date", "home_team", "away_team", 
    "home_score", "away_score", "tournament", "neutral"
]].copy()

home_data.columns = [
    "date", "team", "opponent",
    "goals_for", "goals_against", "tournament", "neutral"
]

home_data["is_hone"] = 1

away_data = df[[
    "date", "away_team", "home_team", 
    "away_score", "home_score", "tournament", "neutral"
]].copy()

away_data.columns = [
    "date", "team", "opponent",
    "goals_for", "goals_against", "tournament", "neutral"
]

away_data["is_home"] = 0

team_matches = pd.concat([home_data, away_data], ignore_index=True)
team_matches = team_matches.sort_values(["team", "date"]).reset_index(drop=True)

team_matches.head()

,date,team,opponent,goals_for,goals_against,tournament,neutral,is_hone,is_home
0,2012-09-25,Abkhazia,Artsakh,1.0,1.0,Friendly,False,1.0,NaN
1,2012-10-21,Abkhazia,Artsakh,0.0,3.0,Friendly,False,NaN,0.0
2,2013-09-23,Abkhazia,South Ossetia,3.0,0.0,Friendly,False,1.0,NaN
3,2014-06-01,Abkhazia,Occitania,1.0,1.0,CONIFA World Football Cup,True,1.0,NaN
4,2014-06-02,Abkhazia,Sápmi,2.0,1.0,CONIFA World Football Cup,False,NaN,0.0


## Membuat Kolom Hasil dan Poin Tim


In [4]:
def get_team_result(row):
    if row["goals_for"] > row["goals_against"]:
        return "win"
    if row["goals_for"] < row["goals_against"]:
        return "lose"
    else:
        return "draw"

team_matches["team_result"] = team_matches.apply(get_team_result, axis=1)
team_matches["points"] = team_matches["team_result"].map({
    "win": 3,
    "draw": 1,
    "lose": 0
})

team_matches["win"] = (team_matches["team_result"] == "win").astype(int)
team_matches.head()

,date,team,opponent,goals_for,goals_against,tournament,neutral,is_hone,is_home,team_result,points,win
0,2012-09-25,Abkhazia,Artsakh,1.0,1.0,Friendly,False,1.0,NaN,draw,1,0
1,2012-10-21,Abkhazia,Artsakh,0.0,3.0,Friendly,False,NaN,0.0,lose,0,0
2,2013-09-23,Abkhazia,South Ossetia,3.0,0.0,Friendly,False,1.0,NaN,win,3,1
3,2014-06-01,Abkhazia,Occitania,1.0,1.0,CONIFA World Football Cup,True,1.0,NaN,draw,1,0
4,2014-06-02,Abkhazia,Sápmi,2.0,1.0,CONIFA World Football Cup,False,NaN,0.0,win,3,1


## Fungsi Mengambil Statistik Terbaru Tim

In [5]:
def get_latest_team_stats(team_name, team_matches, window=5):
    team_data = team_matches[team_matches["team"] == team_name].copy()

    if team_data.empty:
        raise ValueError(f"Tim '{team_name}' tidak ditemukan di dataset.")

    latest_matches = team_data.sort_values("date").tail(window)

    stats = {
        "goals_for_last5": latest_matches["goals_for"].mean(),
        "goals_against_last5": latest_matches["goals_against"].mean(),
        "points_last5": latest_matches["points"].mean(),
        "win_rate_last5": latest_matches["win"].mean(),
        "matches_before": len(team_data)
    }

    return stats


## Fungsi Probabilitas Pertandingan


In [6]:
def get_match_probabilities(home_team, away_team, tournament="FIFA World Cup", neutral=True):
    home_stats = get_latest_team_stats(home_team, team_matches)
    away_stats = get_latest_team_stats(away_team, team_matches)
    
    input_data = pd.DataFrame([{
        "home_team": home_team,
        "away_team": away_team,
        "tournament": tournament,
        "neutral": neutral,
        
        "home_goals_for_last5": home_stats["goals_for_last5"],
        "home_goals_against_last5": home_stats["goals_against_last5"],
        "home_points_last5": home_stats["points_last5"],
        "home_win_rate_last5": home_stats["win_rate_last5"],
        
        "away_goals_for_last5": away_stats["goals_for_last5"],
        "away_goals_against_last5": away_stats["goals_against_last5"],
        "away_points_last5": away_stats["points_last5"],
        "away_win_rate_last5": away_stats["win_rate_last5"],
        
        "goals_for_diff_last5": home_stats["goals_for_last5"] - away_stats["goals_for_last5"],
        "goals_against_diff_last5": home_stats["goals_against_last5"] - away_stats["goals_against_last5"],
        "points_diff_last5": home_stats["points_last5"] - away_stats["points_last5"],
        "win_rate_diff_last5": home_stats["win_rate_last5"] - away_stats["win_rate_last5"],
        "experience_diff": home_stats["matches_before"] - away_stats["matches_before"]
    }])
    
    probabilities = model.predict_proba(input_data)[0]
    classes = model.classes_
    
    prob_dict = dict(zip(classes, probabilities))
    
    return {
        "home_win": prob_dict.get("home_win", 0),
        "draw": prob_dict.get("draw", 0),
        "away_win": prob_dict.get("away_win", 0)
    }

In [7]:
get_match_probabilities("Argentina", "France")

{'home_win': np.float64(0.44199551517357144),
 'draw': np.float64(0.34392999769441585),
 'away_win': np.float64(0.2140744871320127)}

## Menentukan Grup

In [8]:
groups = {
    "Group A": ["Mexico", "Czech Republic", "South Africa", "South Korea"],
    "Group B": ["Canada", "Bosnia and Herzegovina", "Qatar", "Switzerland"],
    "Group C": ["Brazil", "Haiti", "Morocco", "Scotland"],
    "Group D": ["United States", "Australia", "Paraguay", "Turkey"],
    "Group E": ["Curaçao", "Ecuador", "Germany", "Ivory Coast"],
    "Group F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "Group G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "Group H": ["Cape Verde", "Saudi Arabia", "Spain", "Uruguay"],
    "Group I": ["France", "Norway", "Senegal", "Iraq"],
    "Group J": ["Algeria", "Argentina", "Austria", "Jordan"],
    "Group K": ["Colombia", "Jamaica", "Portugal", "Uzbekistan"],
    "Group L": ["Croatia", "England", "Ghana", "Panama"]
}

## Fungsi Simulasi Satu Pertandingan Grup

In [9]:
def generate_score(outcome):
    if outcome == "draw":
        goal = np.random.choice([0, 1, 2, 3], p=[0.25, 0.40, 0.25, 0.10])
        return goal, goal
    
    loser_goals = np.random.choice([0, 1, 2], p=[0.50, 0.35, 0.15])
    margin = np.random.choice([1, 2, 3], p=[0.70, 0.25, 0.05])
    winner_goals = loser_goals + margin
    
    if outcome == "home_win":
        return winner_goals, loser_goals
    else:
        return loser_goals, winner_goals


def simulate_group_match(team_a, team_b):
    probs = get_match_probabilities(team_a, team_b)
    
    outcomes = ["home_win", "draw", "away_win"]
    probabilities = [
        probs["home_win"],
        probs["draw"],
        probs["away_win"]
    ]
    
    outcome = np.random.choice(outcomes, p=probabilities)
    score_a, score_b = generate_score(outcome)
    
    return {
        "team_a": team_a,
        "team_b": team_b,
        "score_a": score_a,
        "score_b": score_b,
        "outcome": outcome
    }

## Fungsi Simulasi Fase Grup

In [10]:
def create_empty_group_table(teams):
    return pd.DataFrame({
        "team": teams,
        "played": 0,
        "win": 0,
        "draw": 0,
        "loss": 0,
        "goals_for": 0,
        "goals_against": 0,
        "goal_diff": 0,
        "points": 0
    })


def update_group_table(table, match_result):
    team_a = match_result["team_a"]
    team_b = match_result["team_b"]
    score_a = match_result["score_a"]
    score_b = match_result["score_b"]
    
    table.loc[table["team"] == team_a, "played"] += 1
    table.loc[table["team"] == team_b, "played"] += 1
    
    table.loc[table["team"] == team_a, "goals_for"] += score_a
    table.loc[table["team"] == team_a, "goals_against"] += score_b
    
    table.loc[table["team"] == team_b, "goals_for"] += score_b
    table.loc[table["team"] == team_b, "goals_against"] += score_a
    
    if score_a > score_b:
        table.loc[table["team"] == team_a, "win"] += 1
        table.loc[table["team"] == team_b, "loss"] += 1
        table.loc[table["team"] == team_a, "points"] += 3
    elif score_a < score_b:
        table.loc[table["team"] == team_b, "win"] += 1
        table.loc[table["team"] == team_a, "loss"] += 1
        table.loc[table["team"] == team_b, "points"] += 3
    else:
        table.loc[table["team"] == team_a, "draw"] += 1
        table.loc[table["team"] == team_b, "draw"] += 1
        table.loc[table["team"] == team_a, "points"] += 1
        table.loc[table["team"] == team_b, "points"] += 1
    
    table["goal_diff"] = table["goals_for"] - table["goals_against"]
    
    return table


def simulate_single_group(group_name, teams):
    table = create_empty_group_table(teams)
    matches = []
    
    for team_a, team_b in combinations(teams, 2):
        match_result = simulate_group_match(team_a, team_b)
        match_result["group"] = group_name
        matches.append(match_result)
        table = update_group_table(table, match_result)
    
    table["group"] = group_name
    
    table = table.sort_values(
        by=["points", "goal_diff", "goals_for", "win"],
        ascending=False
    ).reset_index(drop=True)
    
    table["position"] = table.index + 1
    
    return table, pd.DataFrame(matches)

## Fungsi Menentukan Tim Lolos dari Fase Grup

In [11]:
def simulate_group_stage(groups):
    all_group_tables = []
    all_group_matches = []
    qualified = []
    third_place_teams = []
    
    for group_name, teams in groups.items():
        table, matches = simulate_single_group(group_name, teams)
        
        all_group_tables.append(table)
        all_group_matches.append(matches)
        
        top_two = table.head(2).copy()
        third_place = table.iloc[[2]].copy()
        
        qualified.append(top_two)
        third_place_teams.append(third_place)
    
    qualified_df = pd.concat(qualified, ignore_index=True)
    third_df = pd.concat(third_place_teams, ignore_index=True)
    
    best_third = third_df.sort_values(
        by=["points", "goal_diff", "goals_for", "win"],
        ascending=False
    ).head(8)
    
    qualified_df = pd.concat([qualified_df, best_third], ignore_index=True)
    
    group_tables_df = pd.concat(all_group_tables, ignore_index=True)
    group_matches_df = pd.concat(all_group_matches, ignore_index=True)
    
    qualified_df = qualified_df.sort_values(
        by=["points", "goal_diff", "goals_for", "win"],
        ascending=False
    ).reset_index(drop=True)
    
    return qualified_df, group_tables_df, group_matches_df

## Fungsi Simulasi Knockout Stage

In [12]:
def simulate_knockout_match(team_a, team_b):
    probs = get_match_probabilities(team_a, team_b)
    
    # Pada knockout tidak boleh seri.
    # Probabilitas draw dibagi rata ke kedua tim.
    prob_a = probs["home_win"] + (probs["draw"] / 2)
    prob_b = probs["away_win"] + (probs["draw"] / 2)
    
    total = prob_a + prob_b
    prob_a = prob_a / total
    prob_b = prob_b / total
    
    winner = np.random.choice([team_a, team_b], p=[prob_a, prob_b])
    
    return winner, prob_a, prob_b


def simulate_knockout_stage(qualified_df):
    current_teams = qualified_df["team"].tolist()
    
    rounds = {}
    
    round_names = [
        "Round of 32",
        "Round of 16",
        "Quarter Final",
        "Semi Final",
        "Final"
    ]
    
    for round_name in round_names:
        matches = []
        winners = []
        
        for i in range(len(current_teams) // 2):
            team_a = current_teams[i]
            team_b = current_teams[-(i + 1)]
            
            winner, prob_a, prob_b = simulate_knockout_match(team_a, team_b)
            winners.append(winner)
            
            matches.append({
                "round": round_name,
                "team_a": team_a,
                "team_b": team_b,
                "prob_team_a": round(prob_a * 100, 2),
                "prob_team_b": round(prob_b * 100, 2),
                "winner": winner
            })
        
        rounds[round_name] = pd.DataFrame(matches)
        current_teams = winners
    
    champion = current_teams[0]
    
    return champion, rounds

## Simulasi Satu Turnamen Penuh


In [13]:
def simulate_full_tournament(groups):
    qualified_df, group_tables_df, group_matches_df = simulate_group_stage(groups)
    champion, rounds = simulate_knockout_stage(qualified_df)
    
    return {
        "champion": champion,
        "qualified_df": qualified_df,
        "group_tables_df": group_tables_df,
        "group_matches_df": group_matches_df,
        "rounds": rounds
    }

## Mengecek Kesesuaian Nama Tim dengan Dataset

In [14]:
dataset_teams = set(team_matches["team"].unique())

group_teams = []
for group_name, teams in groups.items():
    group_teams.extend(teams)

missing_teams = [team for team in group_teams if team not in dataset_teams]

if len(missing_teams) == 0:
    print("Semua nama tim pada grup ditemukan di dataset.")
else:
    print("Ada nama tim yang tidak ditemukan di dataset:")
    for team in missing_teams:
        print("-", team)

Semua nama tim pada grup ditemukan di dataset.


## Menjalankan Banyak Simulasi Turnamen

In [15]:
def run_tournament_simulations(groups, n_simulations=100):
    all_teams = [team for group_teams in groups.values() for team in group_teams]
    
    stage_counter = {
        team: {
            "Round of 32": 0,
            "Round of 16": 0,
            "Quarter Final": 0,
            "Semi Final": 0,
            "Final": 0,
            "Champion": 0
        }
        for team in all_teams
    }
    
    champions = []
    
    for sim in range(n_simulations):
        result = simulate_full_tournament(groups)
        
        qualified_df = result["qualified_df"]
        rounds = result["rounds"]
        champion = result["champion"]
        
        champions.append(champion)
        
        for team in qualified_df["team"]:
            stage_counter[team]["Round of 32"] += 1
        
        for _, row in rounds["Round of 32"].iterrows():
            stage_counter[row["winner"]]["Round of 16"] += 1
        
        for _, row in rounds["Round of 16"].iterrows():
            stage_counter[row["winner"]]["Quarter Final"] += 1
        
        for _, row in rounds["Quarter Final"].iterrows():
            stage_counter[row["winner"]]["Semi Final"] += 1
        
        final_match = rounds["Final"].iloc[0]
        stage_counter[final_match["team_a"]]["Final"] += 1
        stage_counter[final_match["team_b"]]["Final"] += 1
        
        stage_counter[champion]["Champion"] += 1
    
    simulation_df = pd.DataFrame.from_dict(stage_counter, orient="index").reset_index()
    simulation_df = simulation_df.rename(columns={"index": "Team"})
    
    stage_columns = [
        "Round of 32",
        "Round of 16",
        "Quarter Final",
        "Semi Final",
        "Final",
        "Champion"
    ]
    
    for col in stage_columns:
        simulation_df[col + " (%)"] = (simulation_df[col] / n_simulations * 100).round(2)
    
    simulation_df = simulation_df.sort_values(
        by="Champion (%)",
        ascending=False
    ).reset_index(drop=True)
    
    return simulation_df

In [ ]:
simulation_df = run_tournament_simulations(groups, n_simulations=10000)

simulation_df.head(15)

## Visualisasi Peluang Juara

In [ ]:
top_10 = simulation_df.head(10)

plt.figure(figsize=(10, 6))
plt.barh(top_10["Team"], top_10["Champion (%)"])
plt.xlabel("Peluang Juara (%)")
plt.ylabel("Tim")
plt.title("Top 10 Prediksi Peluang Juara World Cup 2026")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

top_10[["Team", "Champion (%)", "Final (%)", "Semi Final (%)", "Quarter Final (%)"]]

## Menyimpan Hasil Simulasi

In [ ]:
output_path = Path("../data/processed/tournament_simulation_results.csv")

output_path.parent.mkdir(parents=True, exist_ok=True)

simulation_df.to_csv(output_path, index=False)

print("Hasil simulasi berhasil disimpan ke:", output_path)